# BB84 key, AES-GCM storage

Alice and Bob agree on a 256-bit key with simulated BB84, then use it with AES-256-GCM. On a simulator with a seeded generator the key is not secret; this shows how the pieces fit together. The code lives in `src/`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import os

from cryptography.exceptions import InvalidTag

from src.bb84_key_exchange import EavesdropperDetected, exchange_key
from src.classical_crypto import decrypt, encrypt

In [2]:
kx = exchange_key(seed=11)
print(f"qubits sent {kx.qubits_sent}, QBER {kx.qber:.1%}, keys match {kx.alice_key == kx.bob_key}")
print(kx.alice_key.hex())

qubits sent 2048, QBER 0.0%, keys match True
7352190f6ddef1b93aaba443d390e4c650f4f878dde7a9820bf3d384eabce7ef


In [3]:
blob = encrypt(kx.alice_key, b"quarterly numbers: do not share")
print("stored:", blob.hex())
print("Bob reads:", decrypt(kx.bob_key, blob))

stored: dd0ae270b54030fc43dc0d05353ce149cc258298392373be7c00b852f97f52f7c891a95ece6fbdb8afd06a9a0de200bfd448281bfc7a71bb61a552
Bob reads: b'quarterly numbers: do not share'


In [4]:
try:
    decrypt(os.urandom(32), blob)
except InvalidTag:
    print("random key: rejected")

try:
    exchange_key(eavesdrop=True, seed=11)
except EavesdropperDetected as err:
    print("with Eve:", err)

random key: rejected
with Eve: QBER 23.8% is above 11%
